In [ ]:
# Install libraries
!pip install transformers evaluate scikit-learn beautifulsoup4 -q

In [ ]:
# Imports
import pandas as pd
import re
import torch
import numpy as np
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
# Check GPU
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Upload dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load dataset
df = pd.read_csv("IMDB Dataset.csv")
df.head()

In [ ]:
# Clean text
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review"].apply(clean_text)

In [ ]:
# Encode labels
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

In [ ]:
# Train / validation / test split
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["clean_review"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

In [ ]:
roberta_train_texts = train_texts
roberta_train_labels = train_labels

roberta_val_texts = val_texts
roberta_val_labels = val_labels

roberta_test_texts = test_texts
roberta_test_labels = test_labels

In [ ]:
# Load RoBERTa tokenizer
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [ ]:
# Tokenization
roberta_train_encodings = roberta_tokenizer(
    roberta_train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

roberta_val_encodings = roberta_tokenizer(
    roberta_val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

roberta_test_encodings = roberta_tokenizer(
    roberta_test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
# Dataset class
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Create dataset objects
roberta_train_dataset = IMDbDataset(roberta_train_encodings, roberta_train_labels)
roberta_val_dataset = IMDbDataset(roberta_val_encodings, roberta_val_labels)
roberta_test_dataset = IMDbDataset(roberta_test_encodings, roberta_test_labels)

In [ ]:
# Load RoBERTa model
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

In [ ]:
# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
roberta_training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",
    fp16=True
)

In [ ]:
# Trainer
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_train_dataset,
    eval_dataset=roberta_val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# Train RoBERTa
roberta_trainer.train()

In [ ]:
# Evaluate on test set
roberta_test_results = roberta_trainer.evaluate(roberta_test_dataset)
print(roberta_test_results)

In [ ]:
# Predictions
roberta_predictions = roberta_trainer.predict(roberta_test_dataset)

roberta_y_pred = np.argmax(roberta_predictions.predictions, axis=1)
roberta_y_true = roberta_predictions.label_ids

In [ ]:
# Classification report
print(classification_report(
    roberta_y_true,
    roberta_y_pred,
    target_names=["negative", "positive"]
))

In [ ]:
# Confusion matrix
print(confusion_matrix(roberta_y_true, roberta_y_pred))

In [ ]:
# Save model
roberta_trainer.save_model("./roberta_imdb_model")
roberta_tokenizer.save_pretrained("./roberta_imdb_model")